In [ ]:
%matplotlib inline

In [28]:
%load_ext rpy2.ipython

In [1]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import func
from sqlalchemy.orm import Query

import src
from src.data.models import Channel
from src.data.models import Comment
from src.data.models import Sentence
from src.data.models import Video

In [46]:
pd.options.display.float_format = "{:.1f}".format

colormap = pd.DataFrame(src.colormap.items(), columns=["channel", "color"])

engine = create_engine(src.PS_ENGINE)

# per Channel

In [3]:
query_all = (
    Query(Channel)
    .join(Video)
    .join(Sentence)
    .group_by(Channel)
    .filter(
        Video.format == "videos",
    )
    .with_entities(
        Channel.channel,
        Channel.channel_follower_count,
        func.count(Video.id.distinct()).label("n_videos"),
        func.count(Sentence.id).label("n_sentences"),
        func.min(Video.datetime_upload).label("first_video"),
    )
    .order_by(Channel.channel_follower_count.desc())
)

query_after = (
    Query(Channel)
    .join(Video)
    .join(Sentence)
    .group_by(Channel)
    .filter(
        Video.is_valid == True,
        Sentence.is_valid == True,
    )
    .with_entities(
        Channel.channel,
        func.count(Video.id.distinct()).label("n_videos_clean"),
        func.count(Sentence.id).label("n_sentences_clean"),
    )
)


with engine.connect() as conn:
    df_all = pd.read_sql(query_all.statement, conn)
    df_after = pd.read_sql(query_after.statement, conn)

df = pd.merge(df_all, df_after, on="channel")
df.channel = df.channel.replace({"BÜNDNIS 90/DIE GRÜNEN": "Grüne"})
df

,channel,channel_follower_count,n_videos,n_sentences,first_video,n_videos_clean,n_sentences_clean
0,AfD-Fraktion Bundestag,388000,5257,318328,2017-12-06,5228,300196
1,AfD TV,250000,1563,160132,2017-06-13,1461,145642
2,DIE LINKE,29000,1499,157693,2008-12-18,436,46840
3,Grüne,26100,1623,119187,2008-05-07,460,47016
4,SPD,24200,1625,191038,2008-05-08,484,67174
5,FDP,23300,1952,159493,2007-09-06,487,36930
6,CDU,21900,2111,142666,2008-08-22,630,54288
7,CSU,5170,730,42547,2008-09-09,145,10463


# per Video

In [4]:
comments_query = (
    Query(Comment.id, func.count(Comment.id))
    .filter(Comment.video_id == Video.id, Comment.is_valid == True)
    .with_entities(func.count(Comment.id))
    .scalar_subquery()
)
query = (
    Query(Video)
    .join(Channel)
    .filter(Video.is_valid == True)
    .with_entities(
        Channel.channel,
        Video.datetime_upload,
        Video.like_count,
        Video.view_count,
        Video.duration,
        comments_query.label("comment_count"),
    )
)

with engine.connect() as conn:
    df = pd.read_sql(query.statement, conn)
df.channel = df.channel.replace({"BÜNDNIS 90/DIE GRÜNEN": "Grüne"})

In [27]:
df.groupby("channel").describe().T

channel                AfD TV  AfD-Fraktion Bundestag       CDU       CSU  \
like_count    count    1467.0                  5230.0     667.0     145.0   
              mean     3624.6                  3876.7      64.3      35.6   
              std      5218.8                  6217.0     234.9      49.9   
              min         0.0                     0.0       0.0       0.0   
              25%      1091.5                   509.5      12.0      12.0   
              50%      2226.0                  1816.0      19.0      21.0   
              75%      4318.0                  4560.5      41.0      45.0   
              max    114061.0                128160.0    3272.0     429.0   
view_count    count    1467.0                  5230.0     667.0     145.0   
              mean    43293.2                 45910.4    9609.8   22332.8   
              std    102184.1                106384.4   67231.3  154230.9   
              min       138.0                   101.0      87.0      89.0   
              25%      8196.5                  4175.2     447.5     426.0   
              50%     17728.0                 14647.5     868.0     800.0   
              75%     41349.5                 44064.2    1887.0    1787.0   
              max   2708077.0               2195987.0 1366292.0 1374514.0   
duration      count    1467.0                  5230.0     667.0     145.0   
              mean      655.9                   435.6     615.2     442.2   
              std       885.0                   829.2    2028.7     805.0   
              min        10.0                    24.0       6.0      19.0   
              25%       193.5                   234.2      73.0      83.0   
              50%       297.0                   280.0     138.0     144.0   
              75%       652.5                   343.0     535.5     250.0   
              max     10018.0                 22156.0   35206.0    4483.0   
comment_count count    1467.0                  5230.0     667.0     145.0   
              mean      398.6                   398.9      41.1       7.5   
              std       678.3                   785.0     158.5      17.8   
              min         0.0                     0.0       0.0       0.0   
              25%        88.0                    36.0       3.0       0.0   
              50%       200.0                   142.0      12.0       0.0   
              75%       469.5                   428.8      35.0       9.0   
              max     13807.0                 17450.0    3141.0     166.0   

channel              DIE LINKE      FDP    Grüne      SPD  
like_count    count      457.0    502.0    461.0    497.0  
              mean       255.5      0.5     79.5    101.8  
              std       1270.8      9.8    578.0    388.2  
              min          0.0      0.0      0.0      0.0  
              25%         49.0      0.0      8.0     21.0  
              50%         88.0      0.0     17.0     40.0  
              75%        146.0      0.0     48.0     81.0  
              max      19970.0    216.0  12171.0   7048.0  
view_count    count      457.0    502.0    461.0    497.0  
              mean     10406.1   5874.0   4528.0   5031.8  
              std      54744.8  28642.2  45437.5  29937.3  
              min        182.0     94.0     44.0    137.0  
              25%        896.0    469.2    364.0    567.0  
              50%       1591.0    980.0    777.0   1145.0  
              75%       3430.0   2812.2   2173.0   2705.0  
              max     769998.0 504371.0 970937.0 562534.0  
duration      count      457.0    502.0    461.0    497.0  
              mean       830.9    620.3    890.8   1082.1  
              std       1210.6   1026.4   2411.6   1675.2  
              min         11.0      8.0     27.0      6.0  
              25%        103.0     77.0    251.0    161.0  
              50%        303.0    194.0    392.0    433.0  
              75%       1106.0    573.2    686.0   1234.0  
              max 

In [81]:
%%R -i df -i colormap -w 1000 -h 600

suppressMessages(library(tidyverse))
library(ggplot2)
library(ggeffects)
library(here)

options(scipen = 999)

cmap <- setNames(colormap$color, colormap$channel)

ggplot(df, aes(x=channel, y=view_count, fill=channel)) +
   geom_violin(draw_quantiles=c(0.25, 0.5, 0.75), alpha=0.7) +
   scale_y_continuous(trans="log10") +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 14
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1)
   ) +
   xlab("Channel") +
   ylab("log10(ViewCount)")


ggsave(here("overleaf/img/view_count_violin.pdf"))

Saving 13.9 x 8.33 in image
